# Part 1 — Start Spark + Load Data

In [1]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, sum as spark_sum
import pyspark.sql.functions as F

Start Spark session

In [2]:
spark = SparkSession.builder \
    .appName("CIS660 Healthcare Mini-Pipeline") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

In [3]:
#checking if spark is running
print("Spark Version:", spark.version)
spark

Spark Version: 4.0.2


In [5]:
import pandas as pd

url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"

# download the dataset using pandas
pdf = pd.read_csv(url)

# converting pandas dataframe to spark dataframe
df = spark.createDataFrame(pdf)

df.show()

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          6|    148|           72|           35|      0|33.6|                   0.627| 50|      1|
|          1|     85|           66|           29|      0|26.6|                   0.351| 31|      0|
|          8|    183|           64|            0|      0|23.3|                   0.672| 32|      1|
|          1|     89|           66|           23|     94|28.1|                   0.167| 21|      0|
|          0|    137|           40|           35|    168|43.1|                   2.288| 33|      1|
|          5|    116|           74|            0|      0|25.6|                   0.201| 30|      0|
|          3|     78|           50|           32|     88|31.0|                   0.248| 26|      1|


In [6]:
# Required outputs for Part 1
print("Raw data - first 5 rows:")
df.show(5, truncate=False)

Raw data - first 5 rows:
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin|BMI |DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|6          |148    |72           |35           |0      |33.6|0.627                   |50 |1      |
|1          |85     |66           |29           |0      |26.6|0.351                   |31 |0      |
|8          |183    |64           |0            |0      |23.3|0.672                   |32 |1      |
|1          |89     |66           |23           |94     |28.1|0.167                   |21 |0      |
|0          |137    |40           |35           |168    |43.1|2.288                   |33 |1      |
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
only showing top 5 rows


In [23]:
print("Raw Schema:")
df.printSchema()

Raw Schema:
root
 |-- Pregnancies: long (nullable = true)
 |-- Glucose: long (nullable = true)
 |-- BloodPressure: long (nullable = true)
 |-- SkinThickness: long (nullable = true)
 |-- Insulin: long (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: long (nullable = true)
 |-- Outcome: long (nullable = true)



# Part 2 — Clinical Data Cleaning

In [8]:
#Replacing 0 → NULL for BMI, BloodPressure, Insulin
df_clean = df.withColumn("bmi_clean",
                         when(col("BMI") == 0, None).otherwise(col("BMI").cast("double")) ) \
             .withColumn("bp_clean",
                         when(col("BloodPressure") == 0, None).otherwise(col("BloodPressure").cast("double")) ) \
             .withColumn("insulin_clean",
                         when(col("Insulin") == 0, None).otherwise(col("Insulin").cast("double")) ) \
             .withColumn("age_clean", col("Age").cast("integer"))

In [9]:
# Keep original columns + new cleaned ones (don't overwrite)
df_clean = df_clean.select(
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "Age", "Outcome",
    "bmi_clean", "bp_clean", "insulin_clean", "age_clean"
)

In [10]:
#2.2 Showing cleaned schema & sample
print("Cleaned Schema:")
df_clean.printSchema()

Cleaned Schema:
root
 |-- Pregnancies: long (nullable = true)
 |-- Glucose: long (nullable = true)
 |-- BloodPressure: long (nullable = true)
 |-- SkinThickness: long (nullable = true)
 |-- Insulin: long (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Age: long (nullable = true)
 |-- Outcome: long (nullable = true)
 |-- bmi_clean: double (nullable = true)
 |-- bp_clean: double (nullable = true)
 |-- insulin_clean: double (nullable = true)
 |-- age_clean: integer (nullable = true)



In [11]:
print("Sample rows with raw vs cleaned columns:")
df_clean.select(
    "BMI", "bmi_clean",
    "BloodPressure", "bp_clean",
    "Insulin", "insulin_clean",
    "Age", "age_clean"
).show(10, truncate=False)

Sample rows with raw vs cleaned columns:
+----+---------+-------------+--------+-------+-------------+---+---------+
|BMI |bmi_clean|BloodPressure|bp_clean|Insulin|insulin_clean|Age|age_clean|
+----+---------+-------------+--------+-------+-------------+---+---------+
|33.6|33.6     |72           |72.0    |0      |NULL         |50 |50       |
|26.6|26.6     |66           |66.0    |0      |NULL         |31 |31       |
|23.3|23.3     |64           |64.0    |0      |NULL         |32 |32       |
|28.1|28.1     |66           |66.0    |94     |94.0         |21 |21       |
|43.1|43.1     |40           |40.0    |168    |168.0        |33 |33       |
|25.6|25.6     |74           |74.0    |0      |NULL         |30 |30       |
|31.0|31.0     |50           |50.0    |88     |88.0         |26 |26       |
|35.3|35.3     |0            |NULL    |0      |NULL         |29 |29       |
|30.5|30.5     |70           |70.0    |543    |543.0        |53 |53       |
|0.0 |NULL     |96           |96.0    |0      |

# Part 3 — Feature Engineering

In [12]:
# 3.1 BMI Category
df_enriched = df_clean.withColumn("bmi_category",
    when(col("bmi_clean").isNull(), "Unknown")
    .when(col("bmi_clean") < 18.5, "Underweight")
    .when((col("bmi_clean") >= 18.5) & (col("bmi_clean") <= 24.9), "Normal")
    .when((col("bmi_clean") >= 25.0) & (col("bmi_clean") <= 29.9), "Overweight")
    .otherwise("Obese")
)

In [13]:
# 3.2 Age Group
# Note: min age in dataset is 21 → no <20 needed
df_enriched = df_enriched.withColumn("age_group",
    when((col("age_clean") >= 21) & (col("age_clean") <= 29), "20-29")
    .when((col("age_clean") >= 30) & (col("age_clean") <= 39), "30-39")
    .when((col("age_clean") >= 40) & (col("age_clean") <= 49), "40-49")
    .otherwise("50+")
)

In [14]:
# Quick check
print("Sample of engineered columns:")
df_enriched.select("Age", "age_clean", "age_group", "bmi_clean", "bmi_category").show(10)

Sample of engineered columns:
+---+---------+---------+---------+------------+
|Age|age_clean|age_group|bmi_clean|bmi_category|
+---+---------+---------+---------+------------+
| 50|       50|      50+|     33.6|       Obese|
| 31|       31|    30-39|     26.6|  Overweight|
| 32|       32|    30-39|     23.3|      Normal|
| 21|       21|    20-29|     28.1|  Overweight|
| 33|       33|    30-39|     43.1|       Obese|
| 30|       30|    30-39|     25.6|  Overweight|
| 26|       26|    20-29|     31.0|       Obese|
| 29|       29|    20-29|     35.3|       Obese|
| 53|       53|      50+|     30.5|       Obese|
| 54|       54|      50+|     NULL|     Unknown|
+---+---------+---------+---------+------------+
only showing top 10 rows


# Part 4 — DataFrame API Aggregation

In [15]:
agg_df = (df_enriched
    .groupBy("age_group")
    .agg(
        avg("bmi_clean").alias("avg_bmi"),
        (spark_sum(when(col("bmi_category") == "Obese", 1).otherwise(0)) / count("*")).alias("obese_rate"),
        count("*").alias("patient_count")
    )
    .orderBy(col("obese_rate").desc())
)

print("Aggregated metrics (DataFrame API) - sorted by obese_rate DESC:")
agg_df.show(truncate=False)

Aggregated metrics (DataFrame API) - sorted by obese_rate DESC:
+---------+------------------+------------------+-------------+
|age_group|avg_bmi           |obese_rate        |patient_count|
+---------+------------------+------------------+-------------+
|40-49    |34.617796610169485|0.7966101694915254|118          |
|30-39    |32.67012195121952 |0.6424242424242425|165          |
|20-29    |32.03762886597938 |0.5681818181818182|396          |
|50+      |30.99885057471264 |0.5280898876404494|89           |
+---------+------------------+------------------+-------------+



# Part 5 — Spark SQL Query

In [17]:
# 5.1 Create temporary view
df_enriched.createOrReplaceTempView("patient_clean")

In [18]:
# 5.2 Spark SQL version (should match DataFrame result)
sql_query = """
SELECT
    age_group,
    AVG(bmi_clean) AS avg_bmi,
    SUM(CASE WHEN bmi_category = 'Obese' THEN 1 ELSE 0 END) / COUNT(*) AS obese_rate,
    COUNT(*) AS patient_count
FROM patient_clean
GROUP BY age_group
ORDER BY obese_rate DESC
"""

result_sql = spark.sql(sql_query)

print("Spark SQL result:")
result_sql.show(truncate=False)

Spark SQL result:
+---------+------------------+------------------+-------------+
|age_group|avg_bmi           |obese_rate        |patient_count|
+---------+------------------+------------------+-------------+
|40-49    |34.617796610169485|0.7966101694915254|118          |
|30-39    |32.67012195121952 |0.6424242424242425|165          |
|20-29    |32.03762886597938 |0.5681818181818182|396          |
|50+      |30.99885057471264 |0.5280898876404494|89           |
+---------+------------------+------------------+-------------+



# Part 6 — Save Gold Output as Parquet

In [19]:
output_path = "/content/gold/bmi_age_metrics"

In [20]:
# Write Parquet (overwrites if exists)
agg_df.write.mode("overwrite").parquet(output_path)

In [21]:
# Proof that files exist
import os

print("Gold output folder contents:")
print(os.listdir("/content/gold/bmi_age_metrics"))
# or more detailed:
!ls -lhR /content/gold/bmi_age_metrics

Gold output folder contents:
['.part-00000-8af484be-e719-4318-8cf8-e6ad1380d5bc-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-8af484be-e719-4318-8cf8-e6ad1380d5bc-c000.snappy.parquet', '_SUCCESS']
/content/gold/bmi_age_metrics:
total 4.0K
-rw-r--r-- 1 root root 1.4K Mar  6 17:24 part-00000-8af484be-e719-4318-8cf8-e6ad1380d5bc-c000.snappy.parquet
-rw-r--r-- 1 root root    0 Mar  6 17:24 _SUCCESS


In [22]:
#  Take Screenshot D here (folder listing with .parquet part files)

print("Gold table saved successfully to:", output_path)

Gold table saved successfully to: /content/gold/bmi_age_metrics
